
# ARC-v0.26 — FEVER-E5 Softmax Score-Channel Audit

## Purpose

Test whether the **index-representation H3abs effect under softmax feedback** survives when feedback weights are recomputed from **shared exact embedding dot-products over the already retrieved candidates**, instead of using the ANN index scores.

This is a **post-primary reviewer-oriented mechanism audit**, not a pristine confirmation. The existing paper has already shown the validation outcomes of the parent FEVER-E5 experiment.

### Frozen comparison

Representation mechanism only:

- low fidelity: IVF-PQ32, nprobe=64
- high fidelity: IVF-SQ8, nprobe=64

Same FEVER-E5 corpus/query row space as ARC-v0.18.

### Two softmax score channels

1. **ANN-score softmax**  
   Uses the scores returned by the ANN index, matching the existing trajectory implementation.

2. **Exact-rescored softmax**  
   Keeps each branch's retrieved candidate IDs, but recomputes score weights as the FP32 dot product between the current normalized query state and the shared normalized corpus embedding for each retrieved candidate.

The exact-rescored condition **does not perform exact nearest-neighbor search**. It removes approximate-score calibration from the feedback weights while preserving ANN candidate selection.

### Frozen policy grid

All 32 existing softmax policies:

- alpha ∈ {0.1, 0.3, 0.5, 0.7}
- k ∈ {5, 20}
- tau ∈ {0.05, 0.1, 0.2, 0.5}

No policy selection from validation outcomes.

### Primary reviewer-oriented estimands

At query level, after averaging across the 32 frozen softmax policies:

- ANN-score representation H3abs
- exact-rescored representation H3abs
- paired exact-rescored minus ANN-score H3abs

Interpretation:

- exact-rescored H3abs remains clearly positive → evidence-selection channel is sufficient for the effect;
- exact-rescored H3abs collapses toward zero → ANN score calibration is a major contributor;
- both remain positive but differ → both channels likely contribute.

All outcomes are retained.


In [ ]:

# Cell 1 — Install/imports and fixed paths
!pip -q install faiss-cpu pyarrow scipy tqdm

from google.colab import drive
from pathlib import Path
from collections import defaultdict
from datetime import datetime, timezone
import hashlib, json, os, time, math

import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

if not Path("/content/drive/MyDrive").is_dir():
    drive.mount("/content/drive")

SEED = 20260824
np.random.seed(SEED)

DIM = 384
TOP_RETRIEVE = 100
UTILITY_K = 10
MAX_ROUNDS = 4
REP_NPROBE = 64
CHECKPOINT_EVERY_QUERIES = 10
BOOTSTRAP_REPS = 10_000

RUN_V018 = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "cross-encoder-fever-replication-v018/20260819-015645"
)

SPLIT_PATH = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-boundary-external-replication-v013/20260817-151852/"
    "v013_boundary_query_split.csv"
)

RAW_FEVER = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/raw-datasets/fever"
)

V026_ROOT = Path(
    "/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/"
    "fever-e5-softmax-score-channel-audit-v026"
)
V026_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = V026_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

faiss.omp_set_num_threads(os.cpu_count() or 1)

print("threads:", faiss.omp_get_max_threads())
print("v0.18:", RUN_V018)
print("v0.26 OUT:", OUT)
print("NOTE: faiss-cpu is used; A100 is not required.")


In [ ]:

# Cell 2 — Helpers + restore frozen split
def sha256_file(path, chunk=16 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def membership_sha(ids):
    return hashlib.sha256(
        "\n".join(sorted(map(str, ids))).encode("utf-8")
    ).hexdigest()

def norm_vec(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float32)
    return x / max(float(np.linalg.norm(x)), eps)

def slope(y):
    y = np.asarray(y, dtype=np.float64)
    x = np.arange(len(y), dtype=np.float64)
    return float(np.polyfit(x, y, 1)[0])

def jacdist(a, b):
    A = set(map(int, a))
    B = set(map(int, b))
    return 1.0 - len(A & B) / max(1, len(A | B))

def ndcg(ids, relevant, k=UTILITY_K):
    ids = np.asarray(ids, dtype=np.int64)[:k]
    gains = np.asarray(
        [1.0 if int(i) in relevant else 0.0 for i in ids],
        dtype=np.float64
    )
    discounts = 1.0 / np.log2(np.arange(2, len(ids) + 2))
    dcg = float(np.sum(gains * discounts))
    m = min(k, len(relevant))
    if m == 0:
        return 0.0
    idcg = float(np.sum(1.0 / np.log2(np.arange(2, m + 2))))
    return dcg / idcg

split_df = pd.read_csv(SPLIT_PATH)
FIT_IDS = split_df.loc[split_df["split"].eq("fit"), "query_id"].astype(str).tolist()
VAL_IDS = split_df.loc[split_df["split"].eq("validation"), "query_id"].astype(str).tolist()

EXPECTED_FIT_SHA = "85c01de943636a080abe10cb18cf5538b705b89f6f042a74c92bac366d89612c"
EXPECTED_VAL_SHA = "5e7bd8e3e5e3f0120b1e93726a19183285624c405ef68b34005c8766a9568b42"

assert len(FIT_IDS) == 3350
assert len(VAL_IDS) == 3316
assert membership_sha(FIT_IDS) == EXPECTED_FIT_SHA
assert membership_sha(VAL_IDS) == EXPECTED_VAL_SHA
assert set(FIT_IDS).isdisjoint(VAL_IDS)

print("Frozen split — PASS")
print("FIT:", len(FIT_IDS), "VAL:", len(VAL_IDS))


In [ ]:

# Cell 3 — Restore FEVER-E5 query embeddings + PQ32/SQ8 indexes
QUERY_IDS_PATH = RUN_V018 / "dev_query_ids.txt"
QUERY_EMB_PATH = RUN_V018 / "dev_query_embeddings.float32.npy"

DEV_QUERY_IDS = QUERY_IDS_PATH.read_text().splitlines()
dev_query_embeddings = np.load(QUERY_EMB_PATH, mmap_mode="r")
DEV_QUERY_INDEX = {str(qid): i for i, qid in enumerate(DEV_QUERY_IDS)}

assert len(DEV_QUERY_IDS) == 6666
assert dev_query_embeddings.shape == (6666, DIM)
assert all(q in DEV_QUERY_INDEX for q in FIT_IDS)
assert all(q in DEV_QUERY_INDEX for q in VAL_IDS)

PQ_PATH = RUN_V018 / "fever-e5-small-v2-ivfpq-nlist4096-m32-nbits8.faiss"
SQ_PATH = RUN_V018 / "fever-e5-small-v2-ivfsq8-nlist4096.faiss"

print("Loading PQ32...")
pq32 = faiss.read_index(str(PQ_PATH))
print("Loading SQ8...")
sq8 = faiss.read_index(str(SQ_PATH))

EXPECTED_DOCS = 5_416_568
assert pq32.ntotal == EXPECTED_DOCS
assert sq8.ntotal == EXPECTED_DOCS

print("Query embeddings:", dev_query_embeddings.shape)
print("PQ32 ntotal:", f"{pq32.ntotal:,}")
print("SQ8 ntotal:", f"{sq8.ntotal:,}")
print("FAISS restore — PASS")


In [ ]:

# Cell 4 — Restore shared corpus embedding row space + DEV qrels
CORPUS_IDS_PATH = RUN_V018 / "corpus_doc_ids.txt"
CORPUS_MEMMAP_PATH = RUN_V018 / "corpus_embeddings.float16.memmap"
DEV_QRELS_PATH = RAW_FEVER / "qrels" / "dev.tsv"

assert CORPUS_IDS_PATH.is_file()
assert CORPUS_MEMMAP_PATH.is_file()
assert DEV_QRELS_PATH.is_file()

qrels_df = pd.read_csv(DEV_QRELS_PATH, sep="\t")
qcol = next(c for c in ["query-id", "query_id", "qid"] if c in qrels_df.columns)
dcol = next(c for c in ["corpus-id", "corpus_id", "doc_id"] if c in qrels_df.columns)
scol = next((c for c in ["score", "relevance", "rel"] if c in qrels_df.columns), None)

qrels_df[qcol] = qrels_df[qcol].astype(str)
qrels_df[dcol] = qrels_df[dcol].astype(str)
qrels_df = qrels_df[qrels_df[qcol].isin(DEV_QUERY_IDS)].copy()

if scol is not None:
    qrels_df[scol] = pd.to_numeric(qrels_df[scol], errors="coerce")
    qrels_df = qrels_df[qrels_df[scol] > 0].copy()

needed_doc_ids = set(qrels_df[dcol])
doc_to_row = {}

with open(CORPUS_IDS_PATH, "r", encoding="utf-8") as f:
    for row, line in enumerate(tqdm(f, total=EXPECTED_DOCS, desc="Mapping qrel docs")):
        doc_id = line.rstrip("\n")
        if doc_id in needed_doc_ids:
            doc_to_row[doc_id] = row

missing_docs = needed_doc_ids - set(doc_to_row)
assert not missing_docs, list(missing_docs)[:20]

QRELS = defaultdict(set)
for _, r in qrels_df.iterrows():
    QRELS[str(r[qcol])].add(int(doc_to_row[str(r[dcol])]))

assert all(q in QRELS and len(QRELS[q]) > 0 for q in VAL_IDS)

corpus_embeddings = np.memmap(
    CORPUS_MEMMAP_PATH,
    dtype=np.float16,
    mode="r",
    shape=(EXPECTED_DOCS, DIM),
)

def fetch_docs(rows):
    rows = np.asarray(rows, dtype=np.int64)
    x = np.asarray(corpus_embeddings[rows], dtype=np.float32)
    x /= np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)
    return x

print("Mapped relevant docs:", len(doc_to_row))
print("Corpus memmap:", corpus_embeddings.shape, corpus_embeddings.dtype)
print("QRELS + ROW SPACE — PASS")


In [ ]:

# Cell 5 — Freeze full 32-softmax policy grid BEFORE v0.26 validation trajectories
ALPHAS = [0.1, 0.3, 0.5, 0.7]
SOFTMAX_K = [5, 20]
TEMPERATURES = [0.05, 0.1, 0.2, 0.5]

POLICIES = []
for alpha in ALPHAS:
    for k in SOFTMAX_K:
        for tau in TEMPERATURES:
            POLICIES.append({
                "method": "softmax",
                "alpha": float(alpha),
                "k": int(k),
                "temperature": float(tau),
                "config_key": (
                    f"softmax-k{k}-a{str(alpha).replace('.', 'p')}-"
                    f"t{str(tau).replace('.', 'p')}"
                ),
            })

assert len(POLICIES) == 32

PROTOCOL = {
    "status": "ARC_V026_POST_PRIMARY_SCORE_CHANNEL_AUDIT_FROZEN",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "parent_run": str(RUN_V018),
    "dataset": "FEVER",
    "encoder": "intfloat/e5-small-v2",
    "validation_query_count": len(VAL_IDS),
    "validation_membership_sha256": membership_sha(VAL_IDS),
    "mechanism": "index_representation",
    "low_retriever": "IVF-PQ32 nprobe=64",
    "high_retriever": "IVF-SQ8 nprobe=64",
    "feedback_updates": MAX_ROUNDS,
    "top_retrieve": TOP_RETRIEVE,
    "utility": "nDCG@10",
    "policy_family": "softmax only",
    "policy_count": len(POLICIES),
    "alphas": ALPHAS,
    "k": SOFTMAX_K,
    "temperatures": TEMPERATURES,
    "channels": {
        "ann_score": "softmax weights from ANN-returned scores",
        "exact_rescore": (
            "same retrieved IDs per branch; softmax weights recomputed from "
            "FP32 dot(current_query_state, shared normalized corpus embedding)"
        ),
    },
    "exact_rescore_is_not_exact_search": True,
    "primary_estimands": [
        "query-averaged ANN-score H3abs",
        "query-averaged exact-rescored H3abs",
        "paired exact-rescored minus ANN-score H3abs",
    ],
    "primary_unit": "query",
    "bootstrap_reps": BOOTSTRAP_REPS,
    "post_primary": True,
    "pristine_confirmation_claim": False,
    "negative_null_or_reversal_retained": True,
    "validation_retuning_allowed": False,
}

PROTOCOL_PATH = OUT / "v026_frozen_score_channel_protocol.json"
PROTOCOL_PATH.write_text(json.dumps(PROTOCOL, indent=2, sort_keys=True))
PROTOCOL_SHA = sha256_file(PROTOCOL_PATH)

pd.DataFrame(POLICIES).to_csv(OUT / "v026_frozen_softmax_policy_grid.csv", index=False)
(OUT / "V026_PROTOCOL_SHA256.txt").write_text(
    f"{PROTOCOL_SHA}  {PROTOCOL_PATH.name}\n"
)

print("Policies:", len(POLICIES))
print("Protocol SHA:", PROTOCOL_SHA)
print("V0.26 PROTOCOL FROZEN — PASS")


In [ ]:

# Cell 6 — Search + two score-channel feedback implementations
def search_index(index, q, nprobe=REP_NPROBE):
    index.nprobe = int(nprobe)
    q = norm_vec(q)[None, :]
    scores, ids = index.search(q, TOP_RETRIEVE)
    valid = ids[0] >= 0
    return scores[0][valid], ids[0][valid]

def feedback_softmax_ann(scores, ids, policy):
    k = int(policy["k"])
    tau = float(policy["temperature"])

    ids = np.asarray(ids[:k], dtype=np.int64)
    s = np.asarray(scores[:k], dtype=np.float64)
    docs = fetch_docs(ids)

    z = s / tau
    z -= z.max()
    w = np.exp(z)
    w /= w.sum()

    return norm_vec((docs * w[:, None]).sum(axis=0))

def feedback_softmax_exact(q_state, ids, policy):
    """
    IMPORTANT:
    This is exact rescoring of already retrieved candidates only.
    It is NOT exact nearest-neighbor search.
    """
    k = int(policy["k"])
    tau = float(policy["temperature"])

    ids = np.asarray(ids[:k], dtype=np.int64)
    docs = fetch_docs(ids)

    q = norm_vec(q_state)
    s_exact = docs @ q

    z = np.asarray(s_exact, dtype=np.float64) / tau
    z -= z.max()
    w = np.exp(z)
    w /= w.sum()

    return norm_vec((docs * w[:, None]).sum(axis=0))

def update_state(q0, feedback, alpha):
    return norm_vec((1.0 - float(alpha)) * q0 + float(alpha) * feedback)

print("Score-channel helpers — PASS")


In [ ]:

# Cell 7 — Coupled representation trajectory for one score channel
def run_rep_pair(qid, policy, channel):
    assert channel in {"ann_score", "exact_rescore"}

    q0 = norm_vec(dev_query_embeddings[DEV_QUERY_INDEX[qid]])
    qL = q0.copy()
    qH = q0.copy()
    rel = QRELS[qid]
    rows = []

    for t in range(MAX_ROUNDS + 1):
        sL, iL = search_index(pq32, qL, REP_NPROBE)
        sH, iH = search_index(sq8, qH, REP_NPROBE)

        if min(len(iL), len(iH)) < max(int(policy["k"]), UTILITY_K):
            raise RuntimeError(
                f"Insufficient retrieval results: qid={qid}, policy={policy}, channel={channel}"
            )

        uL = ndcg(iL, rel)
        uH = ndcg(iH, rel)

        rows.append({
            "query_id": str(qid),
            "mechanism": "representation",
            "score_channel": channel,
            "iteration": int(t),
            "method": "softmax",
            "alpha": float(policy["alpha"]),
            "k": int(policy["k"]),
            "temperature": float(policy["temperature"]),
            "config_key": policy["config_key"],
            "query_state_distance": 1.0 - float(np.dot(norm_vec(qL), norm_vec(qH))),
            "candidate_jaccard_distance": jacdist(
                iL[:TOP_RETRIEVE], iH[:TOP_RETRIEVE]
            ),
            "utility_low": uL,
            "utility_high": uH,
            "signed_utility_gap": uH - uL,
            "abs_utility_gap": abs(uH - uL),
        })

        if t == MAX_ROUNDS:
            break

        if channel == "ann_score":
            fL = feedback_softmax_ann(sL, iL, policy)
            fH = feedback_softmax_ann(sH, iH, policy)
        else:
            fL = feedback_softmax_exact(qL, iL, policy)
            fH = feedback_softmax_exact(qH, iH, policy)

        qL = update_state(q0, fL, policy["alpha"])
        qH = update_state(q0, fH, policy["alpha"])

    return rows

def endpoint_df(traj):
    out = []
    group_cols = [
        "query_id", "mechanism", "score_channel", "method",
        "alpha", "k", "temperature", "config_key"
    ]
    for keys, g in traj.groupby(group_cols, dropna=False, sort=False):
        g = g.sort_values("iteration")
        out.append({
            "query_id": keys[0],
            "mechanism": keys[1],
            "score_channel": keys[2],
            "method": keys[3],
            "alpha": keys[4],
            "k": keys[5],
            "temperature": keys[6],
            "config_key": keys[7],
            "H1_slope": slope(g["query_state_distance"]),
            "H2_slope": slope(g["candidate_jaccard_distance"]),
            "H3_abs_slope": slope(g["abs_utility_gap"]),
            "H3_signed_slope": slope(g["signed_utility_gap"]),
            "R1_abs_gap_final_minus_initial": float(
                g["abs_utility_gap"].iloc[-1] - g["abs_utility_gap"].iloc[0]
            ),
            "final_signed_gap": float(g["signed_utility_gap"].iloc[-1]),
        })
    return pd.DataFrame(out)

# FIT-only smoke; no validation trajectory inspection
for channel in ["ann_score", "exact_rescore"]:
    s = pd.DataFrame(run_rep_pair(FIT_IDS[0], POLICIES[0], channel))
    assert len(s) == MAX_ROUNDS + 1
    assert np.isfinite(
        s[[
            "query_state_distance", "candidate_jaccard_distance",
            "utility_low", "utility_high", "abs_utility_gap"
        ]].to_numpy()
    ).all()

print("FIT-ONLY IMPLEMENTATION SMOKE — PASS")
print("No v0.26 validation trajectory inspected.")



## Before Cell 8

The protocol is now frozen.

This audit is post-primary because the parent FEVER-E5 validation results are already known, but **no v0.26 validation score-channel trajectory should be inspected before the full sweep is complete**.

For compute efficiency, **use a CPU runtime**. `faiss-cpu` is doing the retrieval; an A100 is not required.


In [ ]:

# Cell 8 — Resumable full validation sweep
RUN_DIR = OUT / "validation-score-channel"
RUN_DIR.mkdir(parents=True, exist_ok=True)

CHANNELS = ["ann_score", "exact_rescore"]

expected_rows = (
    len(VAL_IDS)
    * len(POLICIES)
    * len(CHANNELS)
    * (MAX_ROUNDS + 1)
)

print("VALIDATION QUERIES:", len(VAL_IDS))
print("SOFTMAX POLICIES:", len(POLICIES))
print("CHANNELS:", CHANNELS)
print("EXPECTED TRAJECTORY ROWS:", f"{expected_rows:,}")
print("CHECKPOINT DIR:", RUN_DIR)

for start in range(0, len(VAL_IDS), CHECKPOINT_EVERY_QUERIES):
    stop = min(start + CHECKPOINT_EVERY_QUERIES, len(VAL_IDS))
    cp = RUN_DIR / f"traj_{start:05d}_{stop:05d}.parquet"

    if cp.exists():
        print("skip", cp.name)
        continue

    rows = []
    t0 = time.perf_counter()

    for qi in range(start, stop):
        qid = VAL_IDS[qi]
        for policy in POLICIES:
            for channel in CHANNELS:
                rows.extend(run_rep_pair(qid, policy, channel))

    df_cp = pd.DataFrame(rows)
    tmp = cp.with_suffix(".tmp.parquet")
    df_cp.to_parquet(tmp, index=False)
    os.replace(tmp, cp)

    dt = time.perf_counter() - t0
    print(f"wrote {cp.name}: {len(df_cp):,} rows | {dt:.1f}s")

parts = sorted(RUN_DIR.glob("traj_*.parquet"))
assert parts

traj = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

assert len(traj) == expected_rows, (len(traj), expected_rows)
assert traj["query_id"].nunique() == len(VAL_IDS)

TRAJ_PATH = OUT / "v026_validation_score_channel_trajectories.parquet"
traj.to_parquet(TRAJ_PATH, index=False)

endpoints = endpoint_df(traj)
ENDPOINT_PATH = OUT / "v026_validation_score_channel_endpoints.parquet"
endpoints.to_parquet(ENDPOINT_PATH, index=False)

print("Trajectory rows:", f"{len(traj):,}")
print("Endpoint rows:", f"{len(endpoints):,}")
print("FULL V0.26 VALIDATION SWEEP — COMPLETE")


In [ ]:

# Cell 9 — Query-level paired bootstrap
MEASURES = [
    "H1_slope", "H2_slope", "H3_abs_slope",
    "H3_signed_slope", "R1_abs_gap_final_minus_initial"
]

q = (
    endpoints.groupby(["query_id", "score_channel"], as_index=False)[MEASURES]
    .mean()
)

ann = q[q["score_channel"].eq("ann_score")].set_index("query_id").loc[VAL_IDS]
ex = q[q["score_channel"].eq("exact_rescore")].set_index("query_id").loc[VAL_IDS]

assert ann.index.tolist() == ex.index.tolist()

rng = np.random.default_rng(SEED + 2601)

def bootstrap_mean(x, reps=BOOTSTRAP_REPS):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    point = float(x.mean())
    boots = np.empty(reps, dtype=np.float64)
    for b in range(reps):
        idx = rng.integers(0, n, size=n)
        boots[b] = float(x[idx].mean())
    lo, hi = np.quantile(boots, [0.025, 0.975])
    return point, float(lo), float(hi), n

rows = []
for measure in MEASURES:
    a = ann[measure].to_numpy(np.float64)
    e = ex[measure].to_numpy(np.float64)
    d = e - a

    for estimand, arr in [
        (f"ann_score_{measure}", a),
        (f"exact_rescore_{measure}", e),
        (f"exact_minus_ann_{measure}", d),
    ]:
        point, lo, hi, n = bootstrap_mean(arr)
        rows.append({
            "estimand": estimand,
            "mean": point,
            "ci95_low": lo,
            "ci95_high": hi,
            "n_queries": n,
        })

summary = pd.DataFrame(rows)
SUMMARY_PATH = OUT / "v026_score_channel_query_bootstrap.csv"
summary.to_csv(SUMMARY_PATH, index=False)

display(summary.round(6))

def row_for(name):
    return summary[summary["estimand"].eq(name)].iloc[0]

ann_h3 = row_for("ann_score_H3_abs_slope")
ex_h3 = row_for("exact_rescore_H3_abs_slope")
diff_h3 = row_for("exact_minus_ann_H3_abs_slope")

print("\nPRIMARY SCORE-CHANNEL RESULTS")
print("ANN-score H3abs:", ann_h3.to_dict())
print("Exact-rescore H3abs:", ex_h3.to_dict())
print("Exact - ANN H3abs:", diff_h3.to_dict())


In [ ]:

# Cell 10 — Alpha/k/tau structure audit
config_summary = (
    endpoints.groupby(
        ["score_channel", "alpha", "k", "temperature", "config_key"],
        as_index=False
    )["H3_abs_slope"]
    .mean()
)

CONFIG_PATH = OUT / "v026_score_channel_configuration_summary.csv"
config_summary.to_csv(CONFIG_PATH, index=False)

alpha_summary = (
    endpoints.groupby(["score_channel", "alpha"], as_index=False)["H3_abs_slope"]
    .mean()
)
ALPHA_PATH = OUT / "v026_score_channel_alpha_summary.csv"
alpha_summary.to_csv(ALPHA_PATH, index=False)

display(alpha_summary.round(6))


In [ ]:

# Cell 11 — Final interpretation gate
ann_mean = float(ann_h3["mean"])
ann_lo = float(ann_h3["ci95_low"])
ann_hi = float(ann_h3["ci95_high"])

ex_mean = float(ex_h3["mean"])
ex_lo = float(ex_h3["ci95_low"])
ex_hi = float(ex_h3["ci95_high"])

diff_mean = float(diff_h3["mean"])
diff_lo = float(diff_h3["ci95_low"])
diff_hi = float(diff_h3["ci95_high"])

if ex_lo > 0:
    interpretation = (
        "SELECTION_CHANNEL_SUFFICIENT_FOR_POSITIVE_REPRESENTATION_H3ABS"
    )
elif ex_hi < 0:
    interpretation = (
        "EXACT_RESCORE_REVERSES_REPRESENTATION_H3ABS"
    )
else:
    interpretation = (
        "EXACT_RESCORE_REPRESENTATION_H3ABS_UNCERTAIN_OR_NULL"
    )

gate = {
    "status": "ARC_V026_SCORE_CHANNEL_AUDIT_ANALYZED",
    "protocol_sha256": PROTOCOL_SHA,
    "post_primary": True,
    "pristine_confirmation_claim": False,
    "dataset": "FEVER",
    "encoder": "intfloat/e5-small-v2",
    "n_validation_queries": len(VAL_IDS),
    "policy_family": "softmax",
    "policy_count": len(POLICIES),
    "ann_score_H3abs": {
        "mean": ann_mean,
        "ci95": [ann_lo, ann_hi],
    },
    "exact_rescore_H3abs": {
        "mean": ex_mean,
        "ci95": [ex_lo, ex_hi],
    },
    "exact_minus_ann_H3abs": {
        "mean": diff_mean,
        "ci95": [diff_lo, diff_hi],
    },
    "interpretation_gate": interpretation,
    "exact_rescore_is_not_exact_search": True,
    "negative_null_or_reversal_retained": True,
    "validation_retuning_performed": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT_PATH = OUT / "v026_score_channel_final_report.json"
REPORT_PATH.write_text(json.dumps(gate, indent=2, sort_keys=True))

print(json.dumps(gate, indent=2))
print("\nFINAL REPORT:", REPORT_PATH)


In [ ]:

# Cell 12 — Hash final artifacts
artifact_paths = [
    PROTOCOL_PATH,
    OUT / "v026_frozen_softmax_policy_grid.csv",
    TRAJ_PATH,
    ENDPOINT_PATH,
    SUMMARY_PATH,
    CONFIG_PATH,
    ALPHA_PATH,
    REPORT_PATH,
]

hash_rows = []
for p in artifact_paths:
    if p.exists():
        hash_rows.append({
            "file": p.name,
            "bytes": p.stat().st_size,
            "sha256": sha256_file(p),
        })

hash_df = pd.DataFrame(hash_rows)
HASH_PATH = OUT / "V026_SCORE_CHANNEL_ARTIFACT_SHA256.csv"
hash_df.to_csv(HASH_PATH, index=False)

display(hash_df)

print("=" * 88)
print("ARC-v0.26 FEVER-E5 SCORE-CHANNEL AUDIT — COMPLETE")
print("OUT:", OUT)
print("interpretation:", gate["interpretation_gate"])
print("=" * 88)



## Send back

After Cell 12 completes, send:

- `v026_score_channel_query_bootstrap.csv`
- `v026_score_channel_configuration_summary.csv`
- `v026_score_channel_final_report.json`

The most important quantity is:

**exact_rescore_H3abs**

If its 95% query-bootstrap CI remains above zero, the reviewer objection that the representation-side softmax effect is *only* an ANN score-calibration artifact becomes substantially weaker.
